<a href="https://colab.research.google.com/github/dahlgrenmartin/WaveTracer/blob/main/WaveTracer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install and import required libraries

In [21]:
!pip install -q diffusers transformers accelerate ptwt scikit-learn torchvision

In [ ]:
import os
import time
import numpy as np
from sklearn.linear_model import LinearRegression
from PIL import Image

import torch
import torchvision.transforms as T
import ptwt
from huggingface_hub import login
from diffusers import AutoencoderKL

# Download SDXL and Real Image
## Create utilities functions

In [23]:
# Skapa en testbild om ingen finns
if not os.path.exists(INPUT_PATH):
    !wget -q -O real_image.png "https://images.unsplash.com/photo-1544005313-94ddf0286df2?w=1024&q=80" # Standardbild för test
    !wget -q -O sdxl_image.png "https://image-b2.civitai.com/file/civitai-media-cache/fe5a6c3a-44cf-48db-82f0-00605e25edf7/original"

def load_vae(model_id, subfolder, device, dtype_str):
    dtype = torch.float16 if dtype_str == "fp16" else torch.float32
    # subfolder="." skapar problem ibland med diffusers, så vi hanterar det
    kwargs = {"torch_dtype": dtype}
    if subfolder and subfolder != ".":
        kwargs["subfolder"] = subfolder

    vae = AutoencoderKL.from_pretrained(model_id, **kwargs)
    vae.to(device)
    vae.eval()
    return vae

def load_image(path, device):
    img = Image.open(path).convert("RGB")
    # VAE förväntar sig normalt värden i intervallet [-1, 1]
    transform = T.Compose([
        T.Resize((1024, 1024)), # SDXL VAE är optimerad för 1024x1024
        T.ToTensor(),
        T.Normalize([0.5], [0.5])
    ])
    return transform(img).unsqueeze(0).to(device)

def encode(vae, target):
    # Skala latents om VAE config kräver det
    posterior = vae.encode(target.to(vae.dtype)).latent_dist
    z = posterior.sample()
    return z * vae.config.scaling_factor

def decode_raw(vae, z):
    z_scaled = z / vae.config.scaling_factor
    return vae.decode(z_scaled).sample

# Choose image to test

In [27]:
INPUT_PATH = "sdxl_image.png"
#INPUT_PATH = "real_image.png"

# RUN THE LATENT REFINEMENT DETECTION
## Instead of tracing slope on FULL PSNR
## Trace it on wavelet subband where effect is biggest


In [28]:
STEPS = 30
LR = 0.01  # 0.01 for SDXL/Wan/SD1.5 and 0.03 for FLUX.2
MODEL_ID = "madebyollin/sdxl-vae-fp16-fix"
VAE_SUBFOLDER = "" # Ändrad från "." för kompatibilitet med from_pretrained

DTYPE = "fp16"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

WAVELET = "sym4"
WINDOW = 10
CHARB_EPS = 2e-3

def charbonnier(x):
    return torch.sqrt(x * x + CHARB_EPS * CHARB_EPS).mean()

def ptwt_input(x):
    n, c, h, w = x.shape
    return x.float().reshape(n * c, 1, h, w)

def slope(values, window):
    y = np.asarray(values[-window:], dtype=np.float64)
    if len(y) < 2:
        return float("nan")
    x = np.arange(len(y)).reshape(-1, 1)
    return float(LinearRegression().fit(x, y).coef_[0])

def psnr(pred, target):
    mse = (pred - target).square().mean().clamp_min(1e-30)
    return float(-10.0 * torch.log10(mse / 4.0))

def main():
    print(f"Laddar VAE till {DEVICE}...")
    vae = load_vae(MODEL_ID, VAE_SUBFOLDER, DEVICE, DTYPE)
    for p in vae.parameters():
        p.requires_grad_(False)

    target = load_image(INPUT_PATH, DEVICE)
    # Cast z to float32 to prevent 'Attempting to unscale FP16 gradients' error
    z = encode(vae, target).detach().clone().to(torch.float32).requires_grad_(True)
    optimizer = torch.optim.Adam([z], lr=LR)
    scaler = torch.amp.GradScaler("cuda", enabled=(DTYPE == "fp16" and DEVICE == "cuda"))

    history = []
    s = float("nan")
    with torch.no_grad():
        # Cast z to VAE's dtype (float16) for decoding to match VAE model's precision
        baseline = psnr(decode_raw(vae, z.to(vae.dtype)), target)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    print("Startar optimering...")
    for step in range(1, STEPS + 1):
        # Cast z to VAE's dtype (float16) for decoding in the loop as well
        residual = decode_raw(vae, z.to(vae.dtype)) - target
        coeffs = ptwt.wavedec2(ptwt_input(residual), WAVELET, level=3, mode="zero")

        # level-1 detail bands -> refinement loss
        lh1, hl1, hh1 = coeffs[3]
        loss = torch.abs(lh1).mean() + torch.abs(hl1).mean() + torch.abs(hh1).mean()

        # level-3 off-diagonal residual energy -> detector trace (band-PSNR)
        lh3, hl3, _ = coeffs[1]
        e = 0.5 * (lh3.detach().square().mean() + hl3.detach().square().mean())
        history.append(float(-10.0 * torch.log10(e + 1e-30)))

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        s = slope(history, WINDOW)
        print(f"{step}: L3off_band_psnr={history[-1]:+.3f} slope={s:+.4f}", flush=True)

    if DEVICE == "cuda":
        torch.cuda.synchronize()
    verdict = "SYNTHETIC" if s > 0 else "REAL"
    print(f"Slope={s:+.4f}: {verdict}")
    print(f"wall_clock={time.perf_counter() - t0:.3f}s")

if __name__ == "__main__":
    main()

Laddar VAE till cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Startar optimering...
1: L3off_band_psnr=+25.438 slope=+nan
2: L3off_band_psnr=+24.877 slope=-0.5608
3: L3off_band_psnr=+24.275 slope=-0.5815
4: L3off_band_psnr=+23.818 slope=-0.5460
5: L3off_band_psnr=+23.487 slope=-0.4960
6: L3off_band_psnr=+23.240 slope=-0.4461
7: L3off_band_psnr=+23.056 slope=-0.4002
8: L3off_band_psnr=+22.909 slope=-0.3600
9: L3off_band_psnr=+22.785 slope=-0.3255
10: L3off_band_psnr=+22.673 slope=-0.2963
11: L3off_band_psnr=+22.577 slope=-0.2364
12: L3off_band_psnr=+22.493 slope=-0.1837
13: L3off_band_psnr=+22.418 slope=-0.1464
14: L3off_band_psnr=+22.354 slope=-0.1205
15: L3off_band_psnr=+22.294 slope=-0.1021
16: L3off_band_psnr=+22.241 slope=-0.0887
17: L3off_band_psnr=+22.191 slope=-0.0782
18: L3off_band_psnr=+22.146 slope=-0.0695
19: L3off_band_psnr=+22.105 slope=-0.0620
20: L3off_band_psnr=+22.071 slope=-0.0555
21: L3off_band_psnr=+22.037 slope=-0.0501
22: L3off_band_psnr=+22.001 slope=-0.0457
23: L3off_band_psnr=+21.972 slope=-0.0419
24: L3off_band_psnr=+21.